<details><summary style="display:list-item; font-size:16px; color:blue;">Jupyter Help</summary>
    
Having trouble testing your work? Double-check that you have followed the steps below to write, run, save, and test your code!

[Click here for a walkthrough GIF of the steps below](https://static-assets.codecademy.com/Courses/ds-python/jupyter-help.gif)

Run all initial cells to import libraries and datasets. Then follow these steps for each question:

1. Add your solution to the cell with `## YOUR SOLUTION HERE ##`.
2. Run the cell by selecting the `Run` button or the `Shift`+`Enter` keys.
3. Save your work by selecting the `Save` button, the `command`+`s` keys (Mac), or `control`+`s` keys (Windows).
4. Select the `Test Work` button at the bottom left to test your work.

![Screenshot of the buttons at the top of a Jupyter Notebook. The Run and Save buttons are highlighted](https://static-assets.codecademy.com/Paths/ds-python/jupyter-buttons.png)

#### Setup Cell

Run the setup cell below to import the required libraries and load shared components from `langgraph_setup.py`.

`PublishingState` (the shared state schema from Exercise 3, with `edit_log`), `draft_node`, and `llm` (`gpt-4o`) are imported from the setup file. `Literal` enables typed return values for the routing function.

**Note:** The OpenAI API key is already configured in this workspace, so no additional setup is needed. When working in an external environment, refer to the [OpenAI API](https://platform.openai.com/docs/quickstart?language=python) documentation for details on configuring your API key.

In [ ]:
# Local application imports
from langgraph_setup import PublishingState, draft_node, llm

# Third-party imports
from langgraph.graph import StateGraph, START, END
from typing import Literal
import warnings

warnings.filterwarnings("ignore")

#### Checkpoint 1/4: Define Format-Specific Node Functions

Let's create two specialized formatting functions that accept a `state` parameter of type `PublishingState` and handle different content output formats.

- **Blog:** Define a function `format_blog` that formats the `draft` as a blog post (with title, section headings, and conclusion), and returns a dictionary with `final_content` (the formatted post) and `edit_log`.

- **Social:** Define a function `format_social` that condenses the `draft` into a social media post (<300 characters) with 3–5 relevant hashtags, and returns a dictionary with `final_content` (the formatted post) and `edit_log`.
  
**Note:** Don't forget to run the cell and save the notebook before selecting `Test Work`. Open the `Jupyter Help` toggle at the top of the notebook for more details.

In [ ]:
# Define a node that formats the draft as a blog post
def format_blog(state: PublishingState):
    result = llm.invoke(
        f"Format this as a blog post. Add a title, "
        f"section headings, and a conclusion:\n{state['draft']}"
    )
    return {"final_content": result.content,
            "edit_log": ["Formatted as blog"]}

# Define a node that condenses the draft for social media
def format_social(state: PublishingState):
    result = llm.invoke(
        f"Condense this into a social media post (<300 characters). "
        f"Add 3-5 relevant hashtags:\n{state['draft']}"
    )
    return {"final_content": result.content,
            "edit_log": ["Formatted for social media"]}

#### Checkpoint 2/4: Define the Routing Function

Now let's create a routing function that decides which formatting node to use, based on the content type.

- Create a function `route_by_content_type` that takes a `state` of type `PublishingState` and returns either `"format_blog"` or `"format_social"` using a `Literal` type. 

- If the `content_type` in the state is `"social_media"`, return `"format_social"`; otherwise, return `"format_blog"`.

**Note:** Don't forget to run the cell and save the notebook before selecting `Test Work`. Open the `Jupyter Help` toggle at the top of the notebook for more details.

In [ ]:
# Define the routing function
def route_by_content_type(state: PublishingState) -> Literal["format_blog", "format_social"]:
    """Route to the appropriate formatter based on content type."""
    if state["content_type"] == "social_media":
        return "format_social"
    return "format_blog"

#### Checkpoint 3/4: Build the Graph with Conditional Edges

With all node functions and routing logic ready, let's build the graph by adding nodes and defining the execution flow with conditional routing.

- Create a `StateGraph` instance with `PublishingState` as its state type and assign it to `builder`. 

- Add three nodes: `draft` (using `draft_node`), `format_blog`, and `format_social`. 

- Wire the execution flow with an edge from `START` to `draft`, a conditional edge from `draft` using `route_by_content_type`, and edges from `format_blog` and `format_social` to `END`.

**Note:** Don't forget to run the cell and save the notebook before selecting `Test Work`. Open the `Jupyter Help` toggle at the top of the notebook for more details.

In [ ]:
# Initialize the graph with the defined state schema
builder = StateGraph(PublishingState)

# Add nodes to the graph
builder.add_node("draft", draft_node)
builder.add_node("format_blog", format_blog)
builder.add_node("format_social", format_social)

# Connect the nodes using edges
builder.add_edge(START, "draft")
builder.add_conditional_edges("draft", route_by_content_type)
builder.add_edge("format_blog", END)
builder.add_edge("format_social", END)

#### Checkpoint 4/4: Compile and Test the Router

With the graph fully wired, let's compile it and test both content types to see conditional routing in action.

- Call `compile()` on `builder` and assign the result to `routing_pipeline`. 

- A `topic` variable and a loop are already provided — within the loop, invoke `routing_pipeline` with a state dictionary containing `topic`, the loop variable `ct` as `content_type`, and empty values for `draft`, `final_content`, and `edit_log`. 

- Store the output in `result`.

**Note:** Don't forget to run the cell and save the notebook before selecting `Test Work`. Open the `Jupyter Help` toggle at the top of the notebook for more details.

In [ ]:
# Compile the graph into a runnable pipeline
routing_pipeline = builder.compile()

topic = "Tips for improving focus and reducing distractions in a digital world"

for ct in ["blog", "social_media"]:
    print(f"\n{'=' * 100}")
    print(f"Content Type: {ct.upper()}")
    print("=" * 100)
    
    # Invoke the pipeline with the initial state
    result = routing_pipeline.invoke({
        "topic": topic, "content_type": ct,
        "draft": "", "final_content": "", "edit_log": []
    })
    
    print(f"Edit Log: {result['edit_log']}")
    print(f"\nContent Preview:\n{result['final_content'][:500]}...")

<details> 

<summary><strong>Summary</strong></summary> 

In this exercise, we added conditional routing to make the pipeline adaptive. The graph selects the next node based on the current state. The routing function chooses the node, and add_conditional_edges manages the flow.

</details>